# Forex RL experiment

This notebook is a thin experiment interface. Core data, feature, scaling, environment, evaluation, and walk-forward logic lives in `forex_rl`. Training remains an explicit user action and is intentionally not run here.

In [ ]:
# In a fresh environment: pip install -e .[research]
from forex_rl import (ExperimentConfig, ForexEnv, R_metrics, ZScoreScaler,
                      build_walkforward_folds, compute_indicators,
                      daily_mtm_from_trades, download_eurusd, make_scaled_frames)

config = ExperimentConfig()
feature_cols = [
    'ema20', 'rsi', 'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_middle', 'bb_lower', 'atr_n', 'ret',
]
price_cols = ['open', 'high', 'low', 'close', 'atr']

In [ ]:
# Downloading data is opt-in; this cell performs no model training.
raw = download_eurusd(config.symbol, config.start_date, config.end_date)
data = compute_indicators(raw)
assert all(column in data for column in feature_cols + price_cols)
folds = build_walkforward_folds(
    data, config.min_train_months, config.val_months, config.test_months,
    config.step_months, config.use_rolling, config.rolling_train_months,
)
print(f'Rows: {len(data):,}; walk-forward folds: {len(folds)}')

In [ ]:
# Example of leakage-safe preparation for one fold.
# The endpoint overlap in .loc slices is retained from the original methodology.
fold = folds[0]
train = data.loc[fold['train_start']:fold['train_end']].copy()
validation = data.loc[fold['val_start']:fold['val_end']].copy()
train_features, validation_features, scaler = make_scaled_frames(train, validation, feature_cols)
env = ForexEnv(train[price_cols], train_features, window_size=config.window_size,
               position_frac=config.position_risk_frac, sl_atr_mult=config.sl_atr_mult,
               tp_atr_mult=config.tp_atr_mult, max_bars_in_trade=config.max_bars_in_trade,
               transaction_cost=config.transaction_cost, slippage=config.slippage,
               invalid_action_penalty=config.invalid_action_penalty, dd_penalty=config.dd_penalty,
               turnover_penalty=config.turnover_penalty, seed=config.random_seed)
observation, info = env.reset(seed=config.random_seed)
observation.shape, info

## Training and reporting

Install the `research` extra and build the SB3/Gym adapter in a dedicated experiment script before training. Do not treat the former 400,000-step ‘smoke test’ as lightweight validation. The package preserves the original event ordering and known characterization issues documented in `docs/STEP1_CURRENT_IMPLEMENTATION_MAP.md`.